In [1]:
from tqdm.auto import tqdm
import numpy as np
from dotenv import load_dotenv
import importlib
import os
from evaluation_utils import map_progress, calc_total_price, llm_structured_retry, RAGWithUsage, calc_price
import sys
from embedder import Embedder
import pandas as pd
from pydantic import BaseModel, Field
from ingest import build_index
from rag_helper import RAGBase, llm_client
import json
from gitsource import chunk_documents
from minsearch import VectorSearch, Index

from agent import faq_agent, SearchDeps
from ingest import build_index, load_faq_data

load_dotenv()

True

Logfire project URL: https://logfire-us.pydantic.dev/dr-mitzi/starter-project


In [17]:
documents = load_faq_data()
index = build_index(documents)

In [18]:
# Inject the index into the agent via the dependency container
deps = SearchDeps(index=index)

In [20]:
# Ask a question. run_sync blocks until the agent is done;
# the agent may call search multiple times before answering.
question = 'I just discovered the course. Can I join it?'
result = await faq_agent.run(question, deps=deps)
print(result.output)

17:47:14.611 faq_agent run
17:47:14.619   chat gpt-5.4-mini
17:47:17.200   running tool: search
17:47:17.220   chat gpt-5.4-mini
17:47:18.346   running tool: search
17:47:18.366   chat gpt-5.4-mini
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted.

If you’d like, I can also help with other course logistics. Are there other areas you want to explore?


In [23]:
# Ask a question. run_sync blocks until the agent is done;
# the agent may call search multiple times before answering.
question = 'How do I run Ollama locally?'
result = await faq_agent.run(question, deps=deps)
print(result.output)

17:52:15.961 faq_agent run
17:52:15.964   chat gpt-5.4-mini
17:52:16.997   running tool: search
17:52:17.013   chat gpt-5.4-mini
17:52:18.136   running tool: search
17:52:18.165   chat gpt-5.4-mini
To run Ollama locally, the FAQ says:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. Start a local model in your terminal:
```bash
ollama run llama3
```
This downloads the LLaMA 3 model, starts it locally, and opens a chat interface.

3. Test that the local server is running:
```bash
curl http://localhost:11434
```
You should get a JSON response with models.

4. If you want to use it from Python:
```bash
pip install ollama
```

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])
```

If you hit a “

In [22]:
# Ask a question. run_sync blocks until the agent is done;
# the agent may call search multiple times before answering.
question = 'Do I have to submit homework to get a certificate?'
result = await faq_agent.run(question, deps=deps)
print(result.output)

17:48:21.736 faq_agent run
17:48:21.740   chat gpt-5.4-mini
17:48:23.034   running tool: search
17:48:23.049   chat gpt-5.4-mini
No — homework is not required to get a certificate.

For the course FAQ: you need to complete the capstone project and the required peer reviews to get the certificate. Homework is recommended and counts toward your leaderboard rank, but it is not mandatory for the certificate.

If you want, I can also clarify the difference between homework, the capstone, and peer reviews. Are there other areas you want to explore?


In [3]:
!python logfire_pipeline.py

Traceback (most recent call last):
  File "C:\Users\ThinkPad_T490\AppData\Local\Programs\Python\Python313\Lib\site-packages\dlt\extract\pipe_iterator.py", line 274, in _get_source_item
    pipe_item = next(gen)
  File "C:\Users\ThinkPad_T490\Desktop\CV\Python\llm-zoomcamp-code\llm-zoomcamp-hw\logfire_pipeline.py", line 28, in logfire_records
    print(f"\U0001f4ca Found {len(rows)} rows with {len(columns)} columns")
    ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ThinkPad_T490\AppData\Local\Programs\Python\Python313\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f4ca' in position 0: character maps to <undefined>

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\ThinkPad_T490\AppDa

In [5]:
import duckdb
conn = duckdb.connect('logfire_pipeline.duckdb')
result = conn.execute("""
    SELECT COUNT(*) FROM information_schema.tables 
    WHERE table_schema = 'agent_traces'
""").fetchall()
print(result)

[(3,)]


In [46]:
# Get all columns from all tables in the schema
columns = conn.execute("""
    SELECT 
        table_schema,
        table_name, 
        column_name
    FROM information_schema.columns 
--  WHERE table_schema = 'agent_traces'
    ORDER BY table_name, ordinal_position
""").fetchdf()

print(columns)

    table_schema           table_name          column_name
0   agent_traces           _dlt_loads              load_id
1   agent_traces           _dlt_loads          schema_name
2   agent_traces           _dlt_loads               status
3   agent_traces           _dlt_loads          inserted_at
4   agent_traces           _dlt_loads  schema_version_hash
5   agent_traces  _dlt_pipeline_state              version
6   agent_traces  _dlt_pipeline_state       engine_version
7   agent_traces  _dlt_pipeline_state        pipeline_name
8   agent_traces  _dlt_pipeline_state                state
9   agent_traces  _dlt_pipeline_state           created_at
10  agent_traces  _dlt_pipeline_state         version_hash
11  agent_traces  _dlt_pipeline_state         _dlt_load_id
12  agent_traces  _dlt_pipeline_state              _dlt_id
13  agent_traces         _dlt_version              version
14  agent_traces         _dlt_version       engine_version
15  agent_traces         _dlt_version          inserted_

In [47]:
# Show ALL tables in the database, not just agent_traces schema
all_tables = conn.execute("""
    SELECT table_schema, table_name 
    FROM information_schema.tables 
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name
""").fetchdf()

print(all_tables)

   table_schema           table_name
0  agent_traces           _dlt_loads
1  agent_traces  _dlt_pipeline_state
2  agent_traces         _dlt_version


In [25]:
import requests

In [35]:
READ_TOKEN = os.getenv('LOGFIRE_READ_TOKEN')

# Your Logfire project is on the US region (based on the URL you shared earlier)
BASE_URL = 'https://logfire-us.pydantic.dev'

query = """
SELECT 
    trace_id,
    span_name,
    start_timestamp,
    attributes->>'gen_ai.usage.input_tokens' AS input_tokens
FROM records
WHERE attributes->>'gen_ai.usage.input_tokens' IS NOT NULL
ORDER BY start_timestamp DESC
LIMIT 50
"""

response = requests.get(
    f'{BASE_URL}/v1/query',
    params={'sql': query},
    headers={'Authorization': f'Bearer {READ_TOKEN}'}
)

data = response.json()
print(data)

{'columns': [{'name': 'trace_id', 'datatype': 'Utf8', 'nullable': False, 'values': ['019f8fad6f5978ed357691e711d3865a', '019f8fad6f5978ed357691e711d3865a', '019f8fad6f5978ed357691e711d3865a', '019f8fa9dc68197f2cb0906da291150c', '019f8fa9dc68197f2cb0906da291150c', '019f8fa9647743ecd6ea59fa69decc1b', '019f8fa9647743ecd6ea59fa69decc1b', '019f8fa9647743ecd6ea59fa69decc1b', '019f8fa8d633bc00683742daf8acf4b7', '019f8fa8d633bc00683742daf8acf4b7', '019f8fa8d633bc00683742daf8acf4b7']}, {'name': 'span_name', 'datatype': 'Utf8', 'nullable': False, 'values': ['chat gpt-5.4-mini', 'chat gpt-5.4-mini', 'chat gpt-5.4-mini', 'chat gpt-5.4-mini', 'chat gpt-5.4-mini', 'chat gpt-5.4-mini', 'chat gpt-5.4-mini', 'chat gpt-5.4-mini', 'chat gpt-5.4-mini', 'chat gpt-5.4-mini', 'chat gpt-5.4-mini']}, {'name': 'start_timestamp', 'datatype': {'Timestamp': ['Microsecond', 'UTC']}, 'nullable': False, 'values': ['2026-07-23T15:52:18.165041Z', '2026-07-23T15:52:17.013261Z', '2026-07-23T15:52:15.964998Z', '2026-07-23

In [41]:
df = pd.DataFrame({
    col['name']: col['values'] 
    for col in data['columns']
})

In [37]:
df

,trace_id,span_name,start_timestamp,input_tokens
0,019f8fad6f5978ed357691e711d3865a,chat gpt-5.4-mini,2026-07-23T15:52:18.165041Z,2199
1,019f8fad6f5978ed357691e711d3865a,chat gpt-5.4-mini,2026-07-23T15:52:17.013261Z,1366
2,019f8fad6f5978ed357691e711d3865a,chat gpt-5.4-mini,2026-07-23T15:52:15.964998Z,285
3,019f8fa9dc68197f2cb0906da291150c,chat gpt-5.4-mini,2026-07-23T15:48:23.049509Z,1076
4,019f8fa9dc68197f2cb0906da291150c,chat gpt-5.4-mini,2026-07-23T15:48:21.740024Z,288
5,019f8fa9647743ecd6ea59fa69decc1b,chat gpt-5.4-mini,2026-07-23T15:47:53.183196Z,2727
6,019f8fa9647743ecd6ea59fa69decc1b,chat gpt-5.4-mini,2026-07-23T15:47:52.086468Z,1500
7,019f8fa9647743ecd6ea59fa69decc1b,chat gpt-5.4-mini,2026-07-23T15:47:51.036233Z,284
8,019f8fa8d633bc00683742daf8acf4b7,chat gpt-5.4-mini,2026-07-23T15:47:18.366154Z,1527
9,019f8fa8d633bc00683742daf8acf4b7,chat gpt-5.4-mini,2026-07-23T15:47:17.220005Z,910


In [38]:
query = """
SELECT COUNT(*) FROM information_schema.tables 
WHERE table_schema = 'agent_traces'
"""

response = requests.get(
    f'{BASE_URL}/v1/query',
    params={'sql': query},
    headers={'Authorization': f'Bearer {READ_TOKEN}'}
)

data = response.json()
print(data)

{'columns': [{'name': 'count(*)', 'datatype': 'Int64', 'nullable': False, 'values': [0]}]}


In [48]:
READ_TOKEN = os.getenv('LOGFIRE_READ_TOKEN')
BASE_URL = 'https://logfire-us.pydantic.dev'

# Test the query directly
query = "SELECT * FROM records LIMIT 5"
response = requests.get(
    f'{BASE_URL}/v1/query',
    params={'sql': query},
    headers={
        'Authorization': f'Bearer {READ_TOKEN}',
        'Accept': 'application/json'
    }
)

print("Status:", response.status_code)
print("Response:", response.json())

Status: 200
Response: {'columns': [{'name': 'created_at', 'datatype': {'Timestamp': ['Microsecond', 'UTC']}, 'nullable': False, 'values': ['2026-07-23T15:47:47.048708Z', '2026-07-23T15:47:49.479840Z', '2026-07-23T15:47:47.048708Z', '2026-07-23T15:47:47.048708Z', '2026-07-23T15:47:49.479840Z']}, {'name': 'start_timestamp', 'datatype': {'Timestamp': ['Microsecond', 'UTC']}, 'nullable': False, 'values': ['2026-07-23T15:47:17.220005Z', '2026-07-23T15:47:18.366154Z', '2026-07-23T15:47:17.200184Z', '2026-07-23T15:47:18.346429Z', '2026-07-23T15:47:14.611004Z']}, {'name': 'end_timestamp', 'datatype': {'Timestamp': ['Microsecond', 'UTC']}, 'nullable': False, 'values': ['2026-07-23T15:47:18.343841Z', '2026-07-23T15:47:19.487462Z', '2026-07-23T15:47:17.217558Z', '2026-07-23T15:47:18.360646Z', '2026-07-23T15:47:19.493979Z']}, {'name': 'duration', 'datatype': 'Float64', 'nullable': True, 'values': [1.1238369, 1.1213072, 0.0173742, 0.0142172, 4.8829753]}, {'name': 'trace_id', 'datatype': 'Utf8', 'nu

In [49]:


# Connect to your database (check the filename!)
conn = duckdb.connect('logfire_pipeline.duckdb')  # This is the default name

# List ALL tables in ALL schemas
all_tables = conn.execute("""
    SELECT table_schema, table_name 
    FROM information_schema.tables 
    WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
    ORDER BY table_schema, table_name
""").fetchdf()

print("All tables in database:")
print(all_tables)

# Try to find any table with 'records'
records_tables = conn.execute("""
    SELECT table_schema, table_name 
    FROM information_schema.tables 
    WHERE table_name LIKE '%record%'
""").fetchdf()

print("\nTables with 'record' in name:")
print(records_tables)

All tables in database:
   table_schema           table_name
0  agent_traces           _dlt_loads
1  agent_traces  _dlt_pipeline_state
2  agent_traces         _dlt_version

Tables with 'record' in name:
Empty DataFrame
Columns: [table_schema, table_name]
Index: []
